# Noise robustness of trained models — v8

Trains the models at one noise level, then tests them across a range of noise levels with no retraining. The
x-axis is the test noise, the y-axis is test accuracy, and there is a line per model, so the different
solutions can be compared for robustness.

Four model conditions, five seeds each, at two hidden units:

| condition | recurrence | training noise |
|---|---|---|
| standard, noise 0.1 | full `W_hh` | 0.1 |
| masked, noise 0.1 | self-connections only | 0.1 |
| standard, no noise | full `W_hh` | 0.0 |
| masked, no noise | self-connections only | 0.0 |

**Design point.** The test trials are generated **once, clean**, and the noise is added on top at each level.
The underlying trials, including the stimulus intensities, are therefore identical at every noise level and
the only thing that changes is the noise, which is what makes the comparison across the x-axis clean. This
matches the generator's own baseline noise, which is added to all channels at all timesteps with the stimulus
laid on top during the window.

Two expectations, set out before looking: accuracy should decay as noise rises, because the signal-to-noise
ratio falls; and the models trained without noise should fall off fastest, because they never had to learn to
filter anything.

Loads `./generated_trials_v8` for the training set only; test trials are generated in-notebook so the noise
can be controlled.

## 1. Setup

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
DATA_DIR = Path("./generated_trials_v8"); OUT_DIR = Path("./noise_robustness_v8"); OUT_DIR.mkdir(exist_ok=True)
T = 50; T_ON, T_OFF = 10, 20

## 2. Trial generator

The v8 specification, with the baseline noise as a knob so we can build a training set at any noise level
and a clean test set to which noise is added afterwards.

In [ ]:
SUBTASKS = ["det_absent","det_auditory_only","det_visual_only",
            "loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
            "loc_multisensory_same_L","loc_multisensory_same_R","det_multisensory",
            "loc_conflict_audL_visR","loc_conflict_audR_visL"]
CONFLICT = ["det_multisensory","loc_conflict_audL_visR","loc_conflict_audR_visL"]
NONCONF  = [s for s in SUBTASKS if s not in CONFLICT]
TRAIN_COUNTS = {"det_absent":600,"det_visual_only":600,"det_auditory_only":1200,
                "loc_auditory_only_L":300,"loc_auditory_only_R":300,"loc_visual_only_L":300,
                "loc_visual_only_R":300,"loc_multisensory_same_L":300,"loc_multisensory_same_R":300,
                "loc_conflict_audL_visR":300,"loc_conflict_audR_visL":300}   # det_multisensory is test-only

def make_trial(sub, noise_std, rng, split):
    X = rng.normal(0.0, noise_std, (4, T)).astype(np.float32) if noise_std > 0 else np.zeros((4, T), np.float32)
    def add(ch, i): X[ch, T_ON:T_OFF] += i
    if   sub == "det_absent": lab = 0
    elif sub == "det_auditory_only": i = rng.uniform(1,3); add(0,i); add(1,i); lab = 1
    elif sub == "det_visual_only":   i = rng.uniform(1,3); add(2,i); add(3,i); lab = 0
    elif sub == "loc_auditory_only_L": add(0, rng.uniform(1,3)); lab = 3
    elif sub == "loc_auditory_only_R": add(1, rng.uniform(1,3)); lab = 2
    elif sub == "loc_visual_only_L":   add(2, rng.uniform(1,3)); lab = 3
    elif sub == "loc_visual_only_R":   add(3, rng.uniform(1,3)); lab = 2
    elif sub == "loc_multisensory_same_L": i = rng.uniform(1,3); add(0,i); add(2,i); lab = 3
    elif sub == "loc_multisensory_same_R": i = rng.uniform(1,3); add(1,i); add(3,i); lab = 2
    elif sub == "det_multisensory": i = rng.uniform(1,3); [add(c,i) for c in range(4)]; lab = 1
    elif sub == "loc_conflict_audL_visR":
        ai = rng.uniform(1,3); vi = rng.uniform(1,3); add(0, ai); add(3, vi)
        lab = (3 if ai > vi else 2) if split == "test" else int(rng.integers(2,4))
    elif sub == "loc_conflict_audR_visL":
        ai = rng.uniform(1,3); vi = rng.uniform(1,3); add(1, ai); add(2, vi)
        lab = (2 if ai > vi else 3) if split == "test" else int(rng.integers(2,4))
    return X, int(lab)

def make_split(split, noise_std, seed=0):
    rng = np.random.default_rng(1000 + seed + (0 if split == "train" else 7))
    Xs, ys, ts = [], [], []
    counts = TRAIN_COUNTS if split == "train" else {s: 100 for s in SUBTASKS}
    for sub, n in counts.items():
        for _ in range(n):
            X, lab = make_trial(sub, noise_std, rng, split)
            Xs.append(X); ys.append(lab); ts.append(sub)
    return {"X": np.stack(Xs), "y": np.array(ys, np.int64), "types": np.array(ts)}

# Clean test set, generated once. Noise is added on top at each level below, so the underlying
# trials and their intensities are identical across the whole x-axis.
test_clean = make_split("test", 0.0)
print("clean test set:", test_clean["X"].shape)

def add_noise(X, noise_std, rng):
    if noise_std <= 0: return X.copy()
    return (X + rng.normal(0.0, noise_std, X.shape)).astype(np.float32)

## 3. Models: standard GRU and masked GRU (self-connections only)

In [ ]:
class StandardGRU(nn.Module):
    def __init__(self, n_in=4, hidden=2, n_out=4):
        super().__init__(); self.H = hidden
        self.gru = nn.GRU(n_in, hidden, batch_first=True); self.readout = nn.Linear(hidden, n_out)
    def forward(self, x):
        x = x.transpose(1, 2); h, _ = self.gru(x); return self.readout(h)

class MaskedGRU(nn.Module):
    def __init__(self, n_in=4, hidden=2, n_out=4):
        super().__init__(); self.H = hidden; s = 1.0/math.sqrt(hidden)
        self.weight_ih = nn.Parameter(torch.empty(3*hidden, n_in).uniform_(-s, s))
        self.weight_hh = nn.Parameter(torch.empty(3*hidden, hidden).uniform_(-s, s))
        self.bias_ih   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.bias_hh   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.readout   = nn.Linear(hidden, n_out)
        self.register_buffer("mask", torch.eye(hidden).repeat(3, 1))
    def masked_hh(self): return self.weight_hh * self.mask
    def forward(self, x):
        x = x.transpose(1, 2); B, Tt, _ = x.shape; H = self.H; Whh = self.masked_hh()
        Wir, Wiz, Win = self.weight_ih[:H], self.weight_ih[H:2*H], self.weight_ih[2*H:]
        Whr, Whz, Whn = Whh[:H], Whh[H:2*H], Whh[2*H:]
        bir, biz, bin_ = self.bias_ih[:H], self.bias_ih[H:2*H], self.bias_ih[2*H:]
        bhr, bhz, bhn = self.bias_hh[:H], self.bias_hh[H:2*H], self.bias_hh[2*H:]
        h = x.new_zeros(B, H); outs = []
        for t in range(Tt):
            xt = x[:, t, :]
            r = torch.sigmoid(xt @ Wir.T + bir + h @ Whr.T + bhr)
            z = torch.sigmoid(xt @ Wiz.T + biz + h @ Whz.T + bhz)
            n = torch.tanh(xt @ Win.T + bin_ + r * (h @ Whn.T + bhn))
            h = (1 - z) * n + z * h; outs.append(self.readout(h))
        return torch.stack(outs, 1)

## 4. Configuration

Each entry of `CONDITIONS` is (label, kind, training noise). The training loop is the slow part; the testing
sweep is cheap.

In [ ]:
HIDDEN = 2
SEEDS = [0, 1, 2, 3, 4]
N_EPOCHS = 50; LR = 1e-3; BATCH = 64
CONDITIONS = [
    ("standard, trained at noise 0.1", "standard", 0.1),
    ("masked, trained at noise 0.1",   "masked",   0.1),
    ("standard, trained no noise",     "standard", 0.0),
    ("masked, trained no noise",       "masked",   0.0),
]
NOISE_LEVELS = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0]
print("models to train:", len(CONDITIONS)*len(SEEDS), " test evaluations:", len(CONDITIONS)*len(SEEDS)*len(NOISE_LEVELS))

## 5. Train the models (once, at their own training noise level)

In [ ]:
def train_model(kind, train_set, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = (MaskedGRU if kind == "masked" else StandardGRU)(4, HIDDEN, 4).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train_set["X"]), torch.from_numpy(train_set["y"])),
                        batch_size=BATCH, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=LR); loss_fn = nn.CrossEntropyLoss()
    for _ in range(N_EPOCHS):
        model.train()
        for Xb, yb in loader:
            lo = model(Xb); B, Tt, C = lo.shape
            loss = loss_fn(lo.reshape(B*Tt, C), yb.unsqueeze(1).expand(B, Tt).reshape(B*Tt))
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); return model

train_sets = {ns: make_split("train", ns) for ns in sorted({c[2] for c in CONDITIONS})}
trained = {}
t0 = time.time()
for label, kind, tr_noise in CONDITIONS:
    trained[label] = []
    for seed in SEEDS:
        trained[label].append(train_model(kind, train_sets[tr_noise], seed))
    print("trained: %-32s (%.1f min elapsed)" % (label, (time.time()-t0)/60))
print("all models trained in %.1f min" % ((time.time()-t0)/60))

# optional: save so the sweep can be re-run without retraining
for label, models in trained.items():
    for seed, m in zip(SEEDS, models):
        torch.save(m.state_dict(), OUT_DIR / ("%s_seed%d.pt" % (label.replace(", ", "_").replace(" ", "-"), seed)))
print("saved model weights to", OUT_DIR)

## 6. Test every model across the noise sweep

No retraining. Each trained model sees the same clean trials with progressively more noise added.

In [ ]:
def accuracy(model, X, y):
    with torch.no_grad():
        pred = model(torch.from_numpy(X))[:, -1, :].argmax(-1).numpy()
    return float((pred == y).mean()), pred

rng_noise = np.random.default_rng(2026)
noisy_sets = {ns: add_noise(test_clean["X"], ns, rng_noise) for ns in NOISE_LEVELS}

results = {label: np.zeros((len(SEEDS), len(NOISE_LEVELS))) for label, _, _ in CONDITIONS}
nonconf_mask = np.isin(test_clean["types"], NONCONF)
results_nc = {label: np.zeros((len(SEEDS), len(NOISE_LEVELS))) for label, _, _ in CONDITIONS}

for label, kind, tr_noise in CONDITIONS:
    for si, model in enumerate(trained[label]):
        for ni, ns in enumerate(NOISE_LEVELS):
            acc, pred = accuracy(model, noisy_sets[ns], test_clean["y"])
            results[label][si, ni] = acc
            results_nc[label][si, ni] = float((pred[nonconf_mask] == test_clean["y"][nonconf_mask]).mean())
    print("swept: %-32s  acc at noise 0.0 -> %.3f, at 1.0 -> %.3f"
          % (label, results[label][:, 0].mean(), results[label][:, -1].mean()))

## 7. The figure: test accuracy as a function of test noise

One line per model condition, mean across seeds with a band for the spread. The dashed vertical line marks
the noise the noisy-trained models saw in training, so anything to the right of it is out of distribution
for them, and everything to the right of zero is out of distribution for the no-noise models.

In [ ]:
style = {
    "standard, trained at noise 0.1": ("black",     "o", "-"),
    "masked, trained at noise 0.1":   ("tab:red",   "s", "-"),
    "standard, trained no noise":     ("0.55",      "o", "--"),
    "masked, trained no noise":       ("tab:orange","s", "--"),
}
fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
for ax, (res, name) in zip(axes, [(results, "all subtasks"), (results_nc, "non-conflict subtasks only")]):
    for label, _, _ in CONDITIONS:
        m = res[label].mean(0); sd = res[label].std(0)
        col, mk, ls = style[label]
        ax.errorbar(NOISE_LEVELS, m, yerr=sd, marker=mk, ls=ls, lw=2.0, capsize=3, color=col, label=label)
        ax.fill_between(NOISE_LEVELS, m - sd, m + sd, color=col, alpha=0.10)
    ax.axhline(0.25, color="gray", ls=":", alpha=0.7, label="4-class chance" if name == "all subtasks" else None)
    ax.axvline(0.1, color="steelblue", ls="--", alpha=0.5)
    ax.text(0.105, 0.02, "training noise (0.1)", color="steelblue", fontsize=8, rotation=90, va="bottom")
    ax.set_xlabel("test noise (baseline noise standard deviation)"); ax.set_ylabel("test accuracy")
    ax.set_ylim(0, 1.02); ax.grid(alpha=0.3); ax.set_title(name)
axes[0].legend(loc="upper right", fontsize=8, frameon=False)
fig.suptitle("Noise robustness: models trained at one noise level, tested across a range (%d hidden units, %d seeds)"
             % (HIDDEN, len(SEEDS)), fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(OUT_DIR / "noise_robustness.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'noise':>7}" + "".join("%28s" % l for l, _, _ in CONDITIONS))
print("-" * (7 + 28*len(CONDITIONS)))
for ni, ns in enumerate(NOISE_LEVELS):
    row = "".join("%28s" % ("%.3f +/- %.3f" % (results[l][:, ni].mean(), results[l][:, ni].std())) for l, _, _ in CONDITIONS)
    print(f"{ns:>7.2f}" + row)

## 8. A single robustness number per model

To compare conditions without reading curves by eye, two summaries per model: the area under the
noise-accuracy curve (higher means more robust across the whole range), and the noise level at which
accuracy first falls below halfway between its clean value and chance, which is a rough half-life.

In [ ]:
def auc(curve): return float(np.trapezoid(curve, NOISE_LEVELS) / (NOISE_LEVELS[-1] - NOISE_LEVELS[0]))
def half_fall(curve):
    clean = curve[0]; thresh = 0.25 + 0.5*(clean - 0.25)
    for ns, a in zip(NOISE_LEVELS, curve):
        if a < thresh: return ns
    return np.nan

print(f"{'condition':>34} {'AUC':>8} {'half-fall noise':>17}")
print("-"*62)
summary = {}
for label, _, _ in CONDITIONS:
    aucs = [auc(results[label][si]) for si in range(len(SEEDS))]
    hfs  = [half_fall(results[label][si]) for si in range(len(SEEDS))]
    summary[label] = (np.mean(aucs), np.nanmean(hfs))
    print(f"{label:>34} {np.mean(aucs):>8.3f} {np.nanmean(hfs):>17.3f}")

a = summary["standard, trained at noise 0.1"][0]; b = summary["masked, trained at noise 0.1"][0]
print("\nmasked vs standard (both trained at 0.1): AUC %.3f vs %.3f, difference %.3f" % (b, a, b - a))
if abs(b - a) < 0.02:
    print("  -> removing cross-connections does not change noise robustness appreciably.")
elif b > a:
    print("  -> the masked model is the more noise-robust of the two.")
else:
    print("  -> cross-connections buy some noise robustness.")

## Notes

- Nothing is retrained across the sweep. Each model is trained once at its own noise level and then tested on
  progressively noisier copies of the same clean trials.
- The no-noise-trained models are out of distribution everywhere except at zero, so a steep fall for them is
  the expected result rather than a surprise; the interesting comparison is masked against standard at
  matched training noise.
- If the cross and the C are genuinely different strategies rather than cosmetic differences, a difference in
  these curves is where it would show up. Pairing the AUC per seed with the geometry measured in notebook 11
  (the angle between the coding axes, and the effective dimensionality) tests that directly.